# Phase 5: Financial Calculations and Guideline Retrieval

This phase separates two responsibilities:

1. Deterministic functions calculate DTI and LTV.
2. A local vector store embeds and retrieves synthetic underwriting rules.

The vector store uses reproducible hashing embeddings rather than an external API. Text becomes a normalized numeric vector; cosine similarity compares a query vector with stored rule vectors. Raw vectors are never placed in the review package.

In [ ]:
# Locate the src-layout project and import the completed earlier phases.
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "underwritingAgent", Path.cwd().parent / "underwritingAgent"]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PDF_ROOT = PROJECT_ROOT / "data" / "realistic_pdfs"
GUIDELINES = PROJECT_ROOT / "data" / "underwriting_guidelines.jsonl"
INTAKE_REFERENCES = ["UW-26-0417-A", "BRK-90831", "WHL-77-2206"]
print(PROJECT_ROOT)


In [ ]:
from underwriting_agent.document_layer import build_document_workflow
from underwriting_agent.borrower_analysis import build_borrower_workflow
from underwriting_agent.intake_packages import resolve_document_paths

def through_phase_3(loan_id):
    paths = resolve_document_paths(PDF_ROOT, loan_id)
    state = build_document_workflow().invoke({
        "loan_id": loan_id,
        "document_paths": [str(path) for path in paths],
        "workflow_status": "INTAKE",
    })
    return build_borrower_workflow().invoke(state)

from underwriting_agent.property_analysis import build_property_workflow
from underwriting_agent.property_research import build_property_research_workflow

def through_phase_4(loan_id):
    state = build_property_workflow().invoke(through_phase_3(loan_id))
    return build_property_research_workflow().invoke(state)


## 1. Test the deterministic formulas in isolation

In [ ]:
from underwriting_agent.calculations_policy import calculate_dti, calculate_ltv

print("DTI:", calculate_dti(monthly_debt=2600, monthly_income=11000))
print("LTV:", calculate_ltv(loan_amount=440000, purchase_price=550000, appraised_value=500000))


## 2. Build and inspect the guideline vector store

At indexing time every rule is embedded once. At query time the query is embedded with the same function, then compared by cosine similarity.

In [ ]:
from underwriting_agent.calculations_policy import LocalGuidelineVectorStore

store = LocalGuidelineVectorStore.from_jsonl(GUIDELINES)
print("Rules indexed:", len(store.rules))
print("Embedding dimensions:", len(store.vectors[0]))
[(rule.rule_id, rule.similarity_score) for rule in store.search("low appraisal and high loan to value", k=3)]


## 3. Run calculation and policy nodes as one subgraph

In [ ]:
from underwriting_agent.calculations_policy import build_calculation_policy_workflow

phase5 = build_calculation_policy_workflow(GUIDELINES)
result = phase5.invoke(through_phase_4("BRK-90831"))
print(result["calculations"].model_dump())
[(rule.rule_id, rule.similarity_score) for rule in result["retrieved_rules"]]


## 4. Evaluate all calculated ratios and retrieved rule IDs

In [ ]:
portfolio = []
for loan_id in INTAKE_REFERENCES:
    result = phase5.invoke(through_phase_4(loan_id))
    portfolio.append({"loan_id": loan_id, "dti": result["calculations"].dti_percent, "ltv": result["calculations"].ltv_percent, "rules": [rule.rule_id for rule in result["retrieved_rules"]]})
portfolio


## Phase 5 handoff

Phase 6 receives exact ratios plus retrieved rule objects. Rules retain IDs so every exception can cite policy provenance.

## Optional production services: OpenAI and Pinecone

The local hashing store remains the offline test backend. In production, `PineconeGuidelineVectorStore` uses OpenAI `text-embedding-3-small` vectors and persists them in a Pinecone serverless dense index. Index dimensions must match the embedding dimensions. Rule text and filterable metadata are upserted under stable `rule_id` values.

Pinecone performs storage and similarity search; it does not generate answers or enforce thresholds. Deterministic code still selects applicable rule IDs and calculates DTI/LTV.

In [ ]:
# Configure .env before enabling this cell's production path.
import os
from dotenv import load_dotenv
from underwriting_agent.integrations import build_integrations_from_env

load_dotenv(PROJECT_ROOT / ".env", override=False)
if os.getenv("GUIDELINE_VECTOR_BACKEND", "local").lower() == "pinecone":
    integrations = build_integrations_from_env(GUIDELINES, dotenv_path=PROJECT_ROOT / ".env")
    pinecone_phase5 = build_calculation_policy_workflow(GUIDELINES, store=integrations.guideline_store)
    print("Pinecone guideline retrieval enabled")
else:
    print("Local guideline vector store enabled (offline default)")
